In [1]:
!pip install pypdf langchain langchain-community chromadb sentence-transformers openai

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma





C:\Users\ITG\AppData\Local\Temp\ipykernel_15696\2168432086.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
import langchain
import langchain_community

print(langchain.__version__)
print(langchain_community.__version__)








1.3.14
0.4.2


In [4]:
!pip install -U langchain-text-splitters

In [5]:
!pip list | findstr langchain

langchain                                1.3.14
langchain-chroma                         1.1.0
langchain-classic                        1.0.8
langchain-community                      0.4.2
langchain-core                           1.5.1
langchain-huggingface                    1.2.2
langchain-openai                         1.4.1
langchain-protocol                       0.0.18
langchain-text-splitters                 1.1.2


In [6]:
!pip install -U langchain-huggingface langchain-chroma

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("Everything imported successfully!")

Everything imported successfully!


In [8]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../data/brochure.pdf")

documents = loader.load()

print(f"Number of pages: {len(documents)}")





Number of pages: 16


In [9]:
print(repr(documents[1].page_content[:200]))

'2\nY ou have been given this leaflet because you requested \ninternational protection (asylum) in this country or in another \nDublin country and the authorities here have reasons to believe \nthat anothe'


In [10]:
print(documents[0].page_content)

1
B
Information for applicants for international protection 
found in a Dublin procedure, pursuant to article 4 of 
Regulation (EU) No 604/2013
“I’m in the  
Dublin procedure  
– what does this mean?”
EN


In [11]:

import re

def clean_text(text):
    # Remove page number if it appears at the beginning
    text = re.sub(r'^\d+\s*\n', '', text)

    # Remove page number if it appears at the end
    text = re.sub(r'\n\s*\d+\s*$', '', text)

    return text

In [12]:
cleaned_text = clean_text(documents[1].page_content)
print(cleaned_text)

Y ou have been given this leaflet because you requested 
international protection (asylum) in this country or in another 
Dublin country and the authorities here have reasons to believe 
that another country might be responsible for examining your 
request. 
We will determine which country is responsible through a 
process established by a European Union law known as the 
‘Dublin’ Regulation. This process is called the ‘Dublin procedure’ . 
This leaflet seeks to answer the most frequent questions you 
might have about this procedure.
If there is anything written here that you do not understand, 
please ask the authorities.
The present leaflet is for information purposes only. Its aim is to provide applicants for 
international protection with the relevant information with respect to the Dublin procedure. 
It does not create/entail in itself rights or legal obligations. The rights and obligations 
of States and persons under the Dublin procedure are such as set out in Regulation (EU) 
6

In [13]:
for i, doc in enumerate(documents):
    print(f"Page {i+1}: {len(doc.page_content)} characters")

Page 1: 203 characters
Page 2: 1174 characters
Page 3: 682 characters
Page 4: 1793 characters
Page 5: 311 characters
Page 6: 2016 characters
Page 7: 1357 characters
Page 8: 1733 characters
Page 9: 1442 characters
Page 10: 1839 characters
Page 11: 1057 characters
Page 12: 1321 characters
Page 13: 27 characters
Page 14: 1815 characters
Page 15: 1237 characters
Page 16: 0 characters


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

documents = loader.load()

for doc in documents:
    doc.page_content = clean_text(doc.page_content)



chunks = text_splitter.split_documents(documents)

print(f"Number of chunks: {len(chunks)}")

Number of chunks: 30


In [15]:
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} -> Page {chunk.metadata['page_label']}")

Chunk 1 -> Page 1
Chunk 2 -> Page 2
Chunk 3 -> Page 2
Chunk 4 -> Page 3
Chunk 5 -> Page 4
Chunk 6 -> Page 4
Chunk 7 -> Page 4
Chunk 8 -> Page 5
Chunk 9 -> Page 6
Chunk 10 -> Page 6
Chunk 11 -> Page 6
Chunk 12 -> Page 7
Chunk 13 -> Page 7
Chunk 14 -> Page 8
Chunk 15 -> Page 8
Chunk 16 -> Page 9
Chunk 17 -> Page 9
Chunk 18 -> Page 10
Chunk 19 -> Page 10
Chunk 20 -> Page 10
Chunk 21 -> Page 11
Chunk 22 -> Page 11
Chunk 23 -> Page 12
Chunk 24 -> Page 12
Chunk 25 -> Page 13
Chunk 26 -> Page 14
Chunk 27 -> Page 14
Chunk 28 -> Page 14
Chunk 29 -> Page 15
Chunk 30 -> Page 15


In [16]:
print(chunks[1].page_content)

Y ou have been given this leaflet because you requested 
international protection (asylum) in this country or in another 
Dublin country and the authorities here have reasons to believe 
that another country might be responsible for examining your 
request. 
We will determine which country is responsible through a 
process established by a European Union law known as the 
‘Dublin’ Regulation. This process is called the ‘Dublin procedure’ . 
This leaflet seeks to answer the most frequent questions you 
might have about this procedure.
If there is anything written here that you do not understand, 
please ask the authorities.
The present leaflet is for information purposes only. Its aim is to provide applicants for 
international protection with the relevant information with respect to the Dublin procedure. 
It does not create/entail in itself rights or legal obligations. The rights and obligations 
of States and persons under the Dublin procedure are such as set out in Regulation (EU)


In [17]:
print(chunks[0].metadata)

{'producer': 'Adobe PDF Library 11.0', 'creator': 'Adobe InDesign CC (Windows)', 'creationdate': '2014-06-12T11:38:03+02:00', 'moddate': '2022-01-27T10:16:24+01:00', 'trapped': '/False', 'source': '../data/brochure.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1'}


In [18]:
for doc in documents:
    doc.metadata = {
        "source": "brochure.pdf",
        "page": doc.metadata["page_label"],
        "document": "Information brochure",
        "topic": "Dublin Regulation"
    }

In [19]:
with open("../data/cleaned_brochure.txt", "w", encoding="utf-8") as f:
    for i, doc in enumerate(documents):
        f.write(f"========== PAGE {i+1} ==========\n\n")
        f.write(doc.page_content)
        f.write("\n\n")

print("Cleaned brochure saved!")

Cleaned brochure saved!


In [20]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)
print("Embedding model loaded!")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded!


In [21]:
test_embedding = embeddings.embed_query(
    "What is the Dublin procedure?"
)

print(len(test_embedding))
print(test_embedding[:5])

384
[-0.007357093971222639, -0.023335935547947884, -0.01358408946543932, -0.03831695765256882, -0.04339839518070221]


In [22]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="../chroma_db"
)

print("Vector database created!")


Vector database created!


In [23]:
collection = vectorstore.get()

print(len(collection["documents"]))

89


In [24]:
results = vectorstore.similarity_search(
    "Which country is responsible for my asylum application?",
    k=3
)

for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("-" * 50)

“How will the authorities establish the country 
responsible for examining my application? “
There are various reasons why a country may be responsible for examining 
your application. These reasons are applied in an order of importance 
given by the law. If one reason is not relevant, the next will be considered, 
and so on. 
The reasons relate to the following factors, in order of importance: 
• you have a family member  (husband or wife, children under the age 
of 18) who has been granted international protection or who is an 
asylum seeker in another Dublin country; 
It is therefore important that you inform us if you have family 
members in another Dublin country, before a first decision is 
made on your asylum request . If you want to be brought together 
in the same country, you and your family member will have to express 
your desire in writing.
• you were previously issued a visa or a residence permit by another 
Dublin country;
{'creator': 'Adobe InDesign CC (Windows)', 'trap

In [25]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [26]:


!pip install langchain-openai openai python-dotenv

In [27]:
from dotenv import load_dotenv
import os

loaded = load_dotenv("../.env")

api_key = os.getenv("OPENAI_API_KEY")

print("Loaded:", loaded)
print("API Key:", api_key[:10] + "..." if api_key else "Not found")

Loaded: True
API Key: sk-proj-Lc...


In [28]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

In [29]:
response = llm.invoke("What is the Dublin Regulation?")

print(response.content)

The Dublin Regulation is a European Union (EU) law that determines which EU member state is responsible for examining an application for asylum seekers seeking international protection. Its primary purpose is to prevent multiple asylum claims in different countries and to ensure that each asylum application is processed by a single member state.

Key points about the Dublin Regulation:

- **Responsibility Criteria:** The regulation sets out criteria to identify the member state responsible for handling an asylum claim. Typically, this is the country where the asylum seeker first entered the EU.

- **Prevention of "Asylum Shopping":** By assigning responsibility to one member state, the regulation aims to prevent asylum seekers from submitting applications in multiple countries to increase their chances of acceptance.

- **Transfers:** If an asylum seeker applies in a member state that is not responsible under the Dublin criteria, they can be transferred to the responsible state.

- **U

In [30]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [31]:
question = "is my information confidential?"

docs = retriever.invoke(question)

print(f"Retrieved {len(docs)} documents")

for i, doc in enumerate(docs, 1):
    print(f"\n===== Document {i} =====")
    print(doc.page_content)
    print("\nMetadata:", doc.metadata)

Retrieved 3 documents

===== Document 1 =====
mind that if you do not agree to let us send your medical information to 
the other country, the other country will not be able to take care of your 
special needs. 
Please note that your medical information will always be handled with 
strict confidentiality by professionals subject to secrecy obligations.
“How long will it take to decide which country will 
treat my application? How long will it take before I 
have my application examined?“
If the authorities in this country decide that we are responsible for 
examining your application for asylum, this means that you may remain in 
this country and have your application examined here.

Metadata: {'creationdate': '2014-06-12T11:38:03+02:00', 'source': '../data/brochure.pdf', 'page': 8, 'producer': 'Adobe PDF Library 11.0', 'creator': 'Adobe InDesign CC (Windows)', 'total_pages': 16, 'moddate': '2022-01-27T10:16:24+01:00', 'page_label': '9', 'trapped': '/False'}

===== Document 2 =====
min

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
You are an AI assistant that helps asylum seekers understand the Dublin Regulation.

Use ONLY the information provided in the context.

If the answer cannot be found in the context, reply:

"I couldn't find that information in the provided documents."

Answer in simple, clear English.

Context:
{context}

Question:
{question}

Answer:
""")

In [33]:
def ask_question(question):
    docs = retriever.invoke(question)

    context = "\n\n".join(doc.page_content for doc in docs)

    messages = prompt.invoke({
        "context": context,
        "question": question
    })

    response = llm.invoke(messages)

    print("Answer:\n")
    print(response.content)

    print("\nSources:")
    for doc in docs:
        print(f"Page {doc.metadata['page_label']}")

    return response.content

In [34]:
answer = ask_question("how long the process of asylum takes?")

print(answer)

Answer:

If the authorities in this country decide to examine your asylum application, you can stay here while they do so. 

If another country is responsible for your application, you will be transferred there within 6 months from when that country accepted responsibility. If you challenge the decision, the transfer will happen within 6 months after a court decides you can be sent. This time can be longer if you run away or are imprisoned. 

If you are in detention, shorter time limits apply.

So, the process can take up to 6 months or more depending on your situation.

Sources:
Page 9
Page 11
Page 11
If the authorities in this country decide to examine your asylum application, you can stay here while they do so. 

If another country is responsible for your application, you will be transferred there within 6 months from when that country accepted responsibility. If you challenge the decision, the transfer will happen within 6 months after a court decides you can be sent. This time can

In [35]:
import sys
print(sys.executable)

c:\Users\ITG\.virtualenvs\Asylum-seekers-assistant-iTIZuJgk\Scripts\python.exe


In [36]:
import sys
print(sys.executable)

c:\Users\ITG\.virtualenvs\Asylum-seekers-assistant-iTIZuJgk\Scripts\python.exe
